# Full Training Pipeline — DrugSafetyHGNN

End-to-end run on the real dataset: random hyperparameter search, retraining the winning config to convergence (early stopping + LR-on-plateau scheduling are already built into `training/train.py`'s `run()`), test-set evaluation, and saving every artifact needed to reproduce the result later.

Produces, under `OUTPUT_DIR`:
- `search_results.json` — every trial's config + val AUROC
- `final/best_model.pt`, `final/final_model.pt` — checkpoints
- `final/training_history.json` — per-epoch train loss / val AUROC / val AUPRC
- `config.json` — the winning hyperparameters (reproducibility)
- `results.json` — final test AUROC / AUPRC + config
- `loss.png`, `auroc.png`, `auprc.png` — training curves

and returns the trained `model` object at the end.

In [1]:
import sys
from pathlib import Path

# Add project root to Python path
sys.path.append(str(Path().resolve().parent))

In [2]:
import json
import random

import matplotlib.pyplot as plt
import torch

from training.train import run
from training.hp_search import SEARCH_SPACE, sample_config

c:\Users\sth3ayush\Desktop\Hackathon\IIMS x Perceptron 2026\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config

`SEARCH_SPACE` (hidden_dim, num_heads, num_layers, lr, batch_size) is imported directly from `training/hp_search.py`, so editing the search space in one place keeps the CLI script (`hp_search.py`) and this notebook in sync.

In [3]:
DATA_PATH = "../graph/heterodata_with_features.pt"
OUTPUT_DIR = Path("../checkpoints/full_run")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_TRIALS = 20
SEARCH_EPOCHS = 15
SEARCH_PATIENCE = 4

FINAL_EPOCHS = 100
FINAL_PATIENCE = 10

VAL_FRAC = 0.1
TEST_FRAC = 0.1
SEED = 42

print("Search space:", json.dumps(SEARCH_SPACE, indent=2))

Search space: {
  "hidden_dim": [
    64,
    128,
    256
  ],
  "num_heads": [
    2,
    4,
    8
  ],
  "num_layers": [
    1,
    2,
    3
  ],
  "lr": [
    0.01,
    0.005,
    0.001,
    0.0005,
    0.0001
  ],
  "batch_size": [
    20000,
    50000,
    100000
  ]
}


## 1. Hyperparameter search

Each trial trains a short, cheap run (`SEARCH_EPOCHS`, early stopping after `SEARCH_PATIENCE` epochs without val AUROC improvement, no checkpoint I/O) purely to rank configs. A failed trial (e.g. an OOM from a large `batch_size`/`hidden_dim` combo) is logged and skipped rather than killing the whole search.

In [4]:
rng = random.Random(SEED)
trial_results = []

for trial_idx in range(1, N_TRIALS + 1):
    config = sample_config(rng)
    trial_dir = OUTPUT_DIR / f"trial_{trial_idx:03d}"
    print(f"=== Trial {trial_idx}/{N_TRIALS}: {config} ===")

    try:
        _, result = run(
            data_path=DATA_PATH,
            output_dir=trial_dir,
            epochs=SEARCH_EPOCHS,
            batch_size=config["batch_size"],
            lr=config["lr"],
            hidden_dim=config["hidden_dim"],
            num_heads=config["num_heads"],
            num_layers=config["num_layers"],
            val_frac=VAL_FRAC,
            test_frac=TEST_FRAC,
            seed=SEED,
            patience=SEARCH_PATIENCE,
            save_artifacts=False,
            verbose=False,
        )
    except Exception as e:
        print(f"  trial failed: {e}")
        continue

    trial_results.append({
        "trial": trial_idx,
        "config": config,
        "best_val_auroc": result["best_val_auroc"],
    })
    print(f"  -> best_val_auroc = {result['best_val_auroc']:.4f}")

assert trial_results, "All trials failed -- check data path and model imports."
trial_results.sort(key=lambda r: r["best_val_auroc"], reverse=True)
(OUTPUT_DIR / "search_results.json").write_text(json.dumps(trial_results, indent=2))
print(f"\n{len(trial_results)}/{N_TRIALS} trials succeeded; results saved to search_results.json")

=== Trial 1/20: {'hidden_dim': 256, 'num_heads': 2, 'num_layers': 1, 'lr': 0.001, 'batch_size': 20000} ===
  -> best_val_auroc = 0.5888
=== Trial 2/20: {'hidden_dim': 64, 'num_heads': 2, 'num_layers': 3, 'lr': 0.01, 'batch_size': 100000} ===
  -> best_val_auroc = 0.5923
=== Trial 3/20: {'hidden_dim': 256, 'num_heads': 8, 'num_layers': 1, 'lr': 0.0001, 'batch_size': 50000} ===
  -> best_val_auroc = 0.5390
=== Trial 4/20: {'hidden_dim': 64, 'num_heads': 2, 'num_layers': 1, 'lr': 0.005, 'batch_size': 20000} ===
  -> best_val_auroc = 0.6021
=== Trial 5/20: {'hidden_dim': 256, 'num_heads': 8, 'num_layers': 1, 'lr': 0.0001, 'batch_size': 20000} ===
  -> best_val_auroc = 0.5508
=== Trial 6/20: {'hidden_dim': 256, 'num_heads': 8, 'num_layers': 3, 'lr': 0.0001, 'batch_size': 50000} ===


KeyboardInterrupt: 

In [ ]:
best = trial_results[0]
print("Best config from search:")
print(json.dumps(best, indent=2))

## 2. Retrain the winning config to convergence

`run()` already handles the two mechanics that matter here:
- **Early stopping** on validation AUROC (`patience=FINAL_PATIENCE`), so it won't blindly run all `FINAL_EPOCHS` if it's stopped improving.
- **LR-on-plateau scheduling** (`ReduceLROnPlateau(mode="max", ...)`), which halves the learning rate when val AUROC stalls.

This time `save_artifacts=True`, so checkpoints and per-epoch history get written to `final_dir`.

In [ ]:
final_dir = OUTPUT_DIR / "final"
model, final_result = run(
    data_path=DATA_PATH,
    output_dir=final_dir,
    epochs=FINAL_EPOCHS,
    batch_size=best["config"]["batch_size"],
    lr=best["config"]["lr"],
    hidden_dim=best["config"]["hidden_dim"],
    num_heads=best["config"]["num_heads"],
    num_layers=best["config"]["num_layers"],
    val_frac=VAL_FRAC,
    test_frac=TEST_FRAC,
    seed=SEED,
    patience=FINAL_PATIENCE,
    save_artifacts=True,
    verbose=True,
)

## 3. Save config + results for reproducibility

In [ ]:
final_config = {
    "hidden_dim": best["config"]["hidden_dim"],
    "num_heads": best["config"]["num_heads"],
    "num_layers": best["config"]["num_layers"],
    "lr": best["config"]["lr"],
    "batch_size": best["config"]["batch_size"],
    "seed": SEED,
    "epochs_run": final_result["stopped_early_at"] or FINAL_EPOCHS,
}
(OUTPUT_DIR / "config.json").write_text(json.dumps(final_config, indent=2))

results = {
    "test_auroc": final_result["test_auroc"],
    "test_auprc": final_result["test_auprc"],
    "best_val_auroc": final_result["best_val_auroc"],
    "stopped_early_at": final_result["stopped_early_at"],
    **final_config,
}
(OUTPUT_DIR / "results.json").write_text(json.dumps(results, indent=2))
print(json.dumps(results, indent=2))

## 4. Training curves

In [ ]:
history = final_result["history"]
epochs_ran = [h["epoch"] for h in history]
train_loss = [h["train_loss"] for h in history]
val_epochs = [h["epoch"] for h in history if "val_auroc" in h]
val_auroc = [h["val_auroc"] for h in history if "val_auroc" in h]
val_auprc = [h["val_auprc"] for h in history if "val_auprc" in h]

plt.figure()
plt.plot(epochs_ran, train_loss)
plt.xlabel("Epoch"); plt.ylabel("Train loss"); plt.title("Training loss")
plt.savefig(OUTPUT_DIR / "loss.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure()
plt.plot(val_epochs, val_auroc)
plt.xlabel("Epoch"); plt.ylabel("Val AUROC"); plt.title("Validation AUROC")
plt.savefig(OUTPUT_DIR / "auroc.png", dpi=150, bbox_inches="tight")
plt.show()

plt.figure()
plt.plot(val_epochs, val_auprc)
plt.xlabel("Epoch"); plt.ylabel("Val AUPRC"); plt.title("Validation AUPRC")
plt.savefig(OUTPUT_DIR / "auprc.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Final model

`model` below is the trained `DrugSafetyHGNN`, already reloaded from its best (highest val-AUROC) checkpoint inside `run()`.

In [ ]:
print("Final model ready.")
print(f"Test AUROC: {final_result['test_auroc']:.4f} | Test AUPRC: {final_result['test_auprc']:.4f}")
print(f"Winning config: {final_config}")
model